# 記事作成ワークフローの実装

このノートブックでは、次の業務プロセスをADKのフレキシブルワークフローで実装します。

1. 企画担当者が記事のテーマを選定する。
2. 調査担当者がテーマに沿った情報を収集して調査レポートにまとめる。
3. 専門のライターが、調査レポートを参考にして原稿を書く。
4. 編集担当者が原稿をレビューして、改善点を指摘する。
5. 原稿の修正とレビューを繰り返して、企画担当者が満足する内容になったら完成とする。

調査担当者、ライター、編集者をエージェントに置き換えて自動化します。

## 事前準備

**[AWW-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[AWW-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[AWW-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[AWW-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[AWW-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk import Event, Workflow
from google.adk.agents.llm_agent import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.planners import BuiltInPlanner
from google.adk.tools import google_search
from google.adk.workflow import DEFAULT_ROUTE
from google.genai.types import (
    GenerateContentConfig, ThinkingConfig, ThinkingLevel,
    HttpOptions, HttpRetryOptions,
)

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[AWW-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

ワークフローの進捗にあわせて結果を表示する `async_output` オプションを追加しています。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message, async_output=False):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
                    if async_output:
                        display(Markdown(f'**[{event["author"]}]**'))
                        display(Markdown(response))
        if not async_output:
            return '\n'.join(result)

**[AWW-07]**

「日本語テキスト出力時のマークダウン表示崩れの防止ルール」と Gemini API のモデル選択をテンプレートとして用意しておきます。


In [6]:
markdown_rule = r'''
# 日本語テキスト出力時のマークダウン表示崩れの防止ルール
マークダウンの修飾記号（太字 `**` や斜体 `*` など）と、日本語の特殊記号や全角文字が隣り合うことで生じる表示崩れを防ぐため、以下のルールを厳守してください。

## 1. 記号と太字の間にスペースを入れる基本ルール
太字記号（`**`）の直前・直後に、鉤括弧や丸括弧、その他の記号が来る場合は、必ず半角スペースを1つ挟んでください。

- **全角鉤括弧（「」）：**
  - 【NG例】`**「重要」**なポイント`
  - 【OK例】`**「重要」** なポイント`
- **全角・半角の丸括弧（（） / ()）：**
  - 【NG例】`**（注意）**事項です`
  - 【OK例】`**（注意）** 事項です`
- **パーセント（% / ％）や数値・単位：**
  - 【NG例】`達成率は**100%**です`
  - 【OK例】`達成率は **100%** です`

## 2. 記号自体のエスケープと厳禁な組み合わせ
- マークダウンの特殊文字として誤認されやすい半角記号（`*`, `_`, `~`, `\` など）をそのまま表示する場合は、隣接するアスタリスク等との間にスペースを設けてください。
- アンダースコア（`_`）は単語の途中の日本語や全角文字と混ざると強力なイタリック・太字パーサに拾われやすいため、強調用以外では原則として直接隣接させないでください。
'''

flash_lite_model = Gemini(
    model='gemini-3.1-flash-lite',
    retry_options=HttpRetryOptions(),
)

flash_model = Gemini(
    model='gemini-3.8-flash',
    retry_options=HttpRetryOptions(),
)

planner = BuiltInPlanner(
    thinking_config=ThinkingConfig(
        thinking_level=ThinkingLevel.LOW,
        include_thoughts=False,
    ),
)

## タスクノードとエージェントノードの定義

**[AWW-08]**

記事のテーマとターゲット層を選定するノードを定義します。

In [7]:
class ArticleInformation(BaseModel):
    """記事の情報"""
    theme: str = Field(description='記事のテーマ')
    audience: str = Field(description='対象読者（ターゲット層）')

instruction = f'''
# タスク
1. ネットに掲載する記事の基本情報をユーザーにヒアリングします。
2. 得られた情報を ArticleInformation にセットします。
3. ArticleInformation の情報がそろったらタスクを終了します。

# 条件
- 必要に応じてユーザーの相談に乗って、google_search で参考情報を収集・提供してください。
- こちらから提案した内容は、ユーザーの確認を得た上で確定してください。

{markdown_rule}
'''

article_information_task = LlmAgent(
    name='article_information_task',
    model=flash_model,
    mode='task',
    output_schema=ArticleInformation,
    description='記事の基本情報を決めるタスク',
    instruction=instruction,
    tools=[google_search],
    planner=planner,
)

async def output_article_information(node_input: ArticleInformation):
    message = f'''
### 記事のテーマとターゲット層を決定しました。
- テーマ: {node_input.theme}
- ターゲット層: {node_input.audience}
'''
    return Event(
        node_output=node_input,
        message=message,
    )

**[AWW-09]**

調査レポートを作成するエージェントノードを定義します。

In [8]:
instruction = f'''
あなたの役割は、記事の執筆に必要な情報を収集して調査レポートにまとめる事です。
- 指定されたテーマとターゲット層の記事を執筆する際に参考となるトピックを5項目選定します。
- それぞれのトピックについて情報を深掘りして、トピックごとに5文以上の長さで調査結果をまとめます。
- トピックの選定と情報の深掘りでは、google_search で最新の情報を検索すること。

{markdown_rule}
'''

research_agent = LlmAgent(
    name='research_agent',
    model=flash_model,
    description='原稿執筆の参考となる調査レポートを作成するエージェント',
    instruction=instruction,
    input_schema=ArticleInformation,
    tools=[google_search],
)

**[AWW-10]**

記事の原稿を執筆するエージェントノードを定義します。

In [9]:
instruction = f'''
あなたの役割は、特定のテーマに関する気軽な読み物記事を書くことです。
記事の「テーマ」「ターゲット層」と、その内容に関連する「調査レポート」が与えられているので、調査レポートに記載の客観的事実に基づいて、信頼性のある記事を書いてください。

# 執筆方針
- トピックに関してある程度の基礎知識がある読者を前提として、数分で気軽に読める内容にしてください。
- 比較的カジュアルで語りかける口調の文章にします。
- 思考過程は出力せずに、最終結果だけを出力します。
- 記事タイトルは付けないで、次の構成で出力します。各セクションタイトルは、内容に合わせてアレンジしてください。
0. 導入：セクションタイトルを付けないで、この記事を読みたくなる導入を1〜2文にまとめます。
1. 概要：トピックの全体像をまとめて簡単に説明します。
2. 最新情報：特に注目したい新しい情報を取り上げます。
3. 実践：トピックに関して、読者自身がやってみるとよさそうな事を1つ紹介します。
4. まとめ

- 各セクションのタイトルはマークダウンヘッダ ## 付けます。必要に応じて箇条書きのマークダウンを使用します。
- それ以外のマークダウンによる装飾は行いません。

# レビュー結果の反映
- レビュアーによる修正ポイントが指摘された際は、直前に書いた記事を指摘に従って書き直します。
- 執筆方針に反する指示がある場合は、レビュアーの指示を優先します。
- これまでの原稿とレビュー結果の履歴がある場合は、その内容も考慮します。

{markdown_rule}
'''

writer_agent = LlmAgent(
    name='writer_agent',
    model=flash_model,
    description='記事の原稿を執筆するエージェント',
    instruction=instruction,
    include_contents='default',
    output_key='article_content',
)

**[AWW-11]**

記事の原稿をレビューするエージェントノードを定義します。

In [10]:
instruction = f'''
あなたの役割は、読み物記事をレビューして、記事の条件にあった内容にするための改善コメントを与える事です。

# 記事の条件
- 記事は、はじめに40文字程度のタイトルがあること。
　今日から役立つ生活情報があって「すぐに読まなきゃ」と読者が感じるタイトルにすること。
　タイトルはマークダウンヘッダ # をつけること。
- タイトルの直後に「なぜいまこのテーマを取り上げるのか」をまとめた導入を加えて、読者にこの記事を読む動機づけを与えます。
- 各セクションのサブタイトルには、絵文字を用いて親しみやすさを出すこと。
- 読者が今日から実践できる具体例が３つ以上紹介されていること。

# 出力形式
- はじめに、記事の良い点を説明します。
- 次に、修正ポイントを箇条書きで出力します。
- これまでの原稿とレビュー結果の履歴がある場合は、その内容も考慮して、直近の原稿をレビューします。

# 注意
- マークダウン表示崩れの防止ルールは自分の出力内容に適用します。レビューの対象項目でありません。

{markdown_rule}
'''

reviewer_agent = LlmAgent(
    name='reviewer_agent',
    model=flash_model,
    description='記事の原稿をレビューするエージェント',
    instruction=instruction,
    include_contents='default',
)

**[AWW-12]**

記事を書き直すかの判断を受け取るタスクノードを定義します。

In [11]:
class HumanCheckResult(BaseModel):
    """記事の書き直しの判断結果"""
    rewrite: bool = Field(description='書き直しの判断結果')

instruction = '''
# タスク
1. レビュー結果に基づいて、記事を書き直すかユーザーに質問します。
2. 得られた情報を HumanCheckResult.rewrite にセットします。
  - 書き直す場合は True
  - 書き直さない場合は False
3. HumanCheckResult.rewrite をセットしたらタスクを終了します。

# 条件
余計な会話はせずに、「レビュー結果に基づいて記事を書き直しますか？」と聞いて、「はい」か「いいえ」の判断を求めてください。
'''

human_check_task = LlmAgent(
    name='human_check_task',
    model=flash_lite_model,
    mode='task',
    output_schema=HumanCheckResult,
    description='記事の書き直しの判断を受け取るタスク',
    instruction=instruction,
)

def process_human_check_result(node_input: HumanCheckResult):
    if node_input.rewrite:
        return Event(
            message='### 記事の書き直しを依頼します。',
            route='rewrite',
        )
    else:
        return Event(
            message='### 記事が完成しました。',
            route='show_last_article',
        )

def show_last_article(ctx, node_input):
    message=f'''
### 完成した記事を表示します。

---

{ctx.state['article_content']}
    '''
    return Event(message=message)

## ワークフローの定義と実行

**[AWW-13]**

ここまでに定義したノードを用いたワークフローを定義します。

In [12]:
article_writing_workflow = Workflow(
    name='article_writing_workflow',
    edges=[
        (
            'START',
            article_information_task, output_article_information,
            research_agent,
            writer_agent, reviewer_agent,
            human_check_task, process_human_check_result,
        ),
        (
            process_human_check_result,
            {
                'rewrite': writer_agent,
                DEFAULT_ROUTE: show_last_article,
            },
        ),
        (
            show_last_article,
        ),
    ],
)

article_writing_app = AdkApp(
    agent=article_writing_workflow,
    app_name='article_writing_app',
)

**[AWW-14]**

ワークフローを実行して、記事を作成します。

In [13]:
chat_client = ChatClient(article_writing_app)

query = '''
秋に向けて、あたらしい趣味を提案する記事を考えています。
「推し活」が流行っているので、これをもっと斬新な切り口でテーマにしたいです。
'''
await chat_client.async_stream_query(query, async_output=True)

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


**[article_information_task]**

秋に向けた「推し活」の新しい切り口ですね。とても面白く、季節感とも相性が良さそうです。

一般的なアイドルやキャラクターの応援とは一味違う、**「秋 × 斬新な推し活」** のテーマとターゲット層（対象読者）の切り口をいくつか考えてみました。

---

### テーマ案のアイデア

1. **「身近な自然・名建築を愛でる『非人間・無機物推し活』」**
   - **内容：** 秋の散歩や紅葉狩りと絡めて、「街のレトロ建築」「純喫茶の椅子」「推し樹木・推し苔」「仏像」など、人以外の特定の対象を愛でて巡る推し活。
   - **ターゲット層：** 休日を静かに楽しみたい20〜40代、人混みやSNS疲れを感じており、マイペースに深くハマれる趣味を探している人。

2. **「旬の味覚をとことん極める『食の推し活（推し品種・推し芋）』」**
   - **内容：** 「食欲の秋」にちなみ、例えば「さつまいもの特定品種（紅はるか推しなど）」や「クラフトビール」「和菓子」など、特定の食材・ジャンルを推して全国のお取り寄せや専門店巡りをする。
   - **ターゲット層：** グルメに関心が高く、日常にプチ贅沢や新しい体験を取り入れたい20〜40代の社会人。

3. **「推しを自給自足する『クリエイティブ推し活（概念・創作）』」**
   - **内容：** 「芸術の秋」をテーマに、既存の推しの概念を香水・カクテル・手芸（刺繍やタフティングなど）で形にしたり、推しをテーマにしたアート作りを体験する。
   - **ターゲット層：** すでに推し活をしているがマンネリ気味の人、ものづくりやクリエイティブな趣味を始めてみたい10〜30代。

4. **「自分を推してご自愛する『セルフ推し活（推し＝自分）』」**
   - **内容：** 「読書の秋・夜長」に向けて、自分自身のファンになって自分を愛でる・労わるケア（ノート術、ご褒美プラン、セルフプロデュース）を推し活風に楽しむ。
   - **ターゲット層：** 日常のモチベーションを上げたい働く世代、自己肯定感やメンタルヘルスに関心がある20〜30代女性など。

---

気になる方向性や、「もっとこういう読者に向けて届けたい」「こういう要素を入れたい」といったイメージはありますでしょうか？

方向性についてご意見をいただき、**「記事のテーマ」** と **「対象読者（ターゲット層）」** を一緒に固めていければと思います！

In [14]:
query = '''
秋の味覚を推し活の視点で捉えるのはいいですね。
ターゲット層の絞り込みが難しそうですが・・・。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_information_task]**

「食の推し活」は、親しみやすさと斬新さのバランスがとても魅力的ですね！

ターゲット層が広がりやすいテーマですので、**「どんな悩みやモチベーションを持っている人に届けたいか」** という視点から、3つの切り口を考えてみました。

---

### ターゲット層の候補案

1. **【手軽なご褒美派】忙しい毎日にちょっとした楽しみがほしい、20代〜30代の働く世代（単身者中心）**
   - **特徴：** 大がかりな趣味を始める時間や体力はないが、季節の変わり目に手軽なワクワク感がほしい層。
   - **記事のアプローチ：** コンビニスイーツの食べ比べや、カフェの「いも・栗・かぼちゃフェア」、1個から買えるお取り寄せなど、**「日常のすきま時間でできる推し品種探し」** を提案。

2. **【推し活疲れ・マイペース派】人間関係やSNSに縛られず、自分のペースで何かに夢中になりたい20代〜40代**
   - **特徴：** アイドルやアニメなどの従来の推し活に少し疲れを感じていたり、周りと競わずに「純粋に偏愛を楽しみたい」と感じている層。
   - **記事のアプローチ：** 「推しは食べてもいい」「誰かと比べず、自分の舌と好みに素直になる」という、**「ストレスフリーで心を満たす癒やしの食推し活」** を提案。

3. **【こだわり・カルチャー派】休日の過ごし方に「体験」や「うんちく」を取り入れたい、食や暮らしに関心の高い20代〜40代**
   - **特徴：** 週末に直売所や専門店へ足を運んだり、品種の違い・産地のストーリーを学んで楽しむのが好きな層。
   - **記事のアプローチ：** さつまいもやぶどうの「品種マニアック図鑑」「専門店巡りマップ」など、**「食欲の秋をマニア目線で深掘りする大人の知的好奇心趣味」** として提案。

---

どのタイプが今回のメディアの雰囲気や発信したいイメージに近いでしょうか？
もちろん「1と2の中間くらい」といった調整も可能ですので、ご感想をお聞かせください！

In [15]:
query = '''
案1を採用。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_information_task]**

「案1」の方向性ですね！日常に気軽に取り入れやすく、共感を得やすい良い切り口だと思います。

記事の基本情報として、以下のように確定させてよろしいでしょうか？

---

* **記事のテーマ：**
  秋の味覚をとことん極める「食の推し活」〜コンビニスイーツや品種の食べ比べで、日常のすきま時間に手軽に楽しむ新しい秋の趣味〜

* **対象読者（ターゲット層）：**
  忙しい毎日にちょっとした楽しみやご褒美がほしい、20代〜30代の働く世代（単身者中心）

---

こちらで問題なければ、基本情報として確定いたします。修正や追加したいポイントなどがあれば教えてください！

In [16]:
query = '''
これで決定します。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**


### 記事のテーマとターゲット層を決定しました。
- テーマ: 秋の味覚をとことん極める「食の推し活」〜コンビニスイーツや品種の食べ比べで、日常のすきま時間に手軽に楽しむ新しい秋の趣味〜
- ターゲット層: 忙しい毎日にちょっとした楽しみやご褒美がほしい、20代〜30代の働く世代（単身者中心）


**[research_agent]**

秋の味覚をとことん極める「食の推し活」をテーマに、20代〜30代の働く単身世代に向けた記事執筆で活用できる **5つのトピック** を選定し、それぞれの調査結果をまとめました。

---

### トピック1：多忙な単身世代に広がる「食の推し活」トレンドの背景
忙しい日々を送る20代〜30代の働く単身層の間で、大がかりなレジャーの代わりに日常のすきま時間で完結する「食の推し活」が注目を集めています。遠出や高額な出費を伴う趣味とは異なり、仕事帰りや休憩時間に立ち寄れる身近な店舗で手に入る秋の味覚は、コストパフォーマンスとタイムパフォーマンスの両面に優れています。さらに近年は単に見た目の映えや万人受けを狙うだけでなく、自分の舌に本当に合う「パーソナルテイスト（自分だけの推し味）」をとことん探求する消費傾向が強まっています。ひとり暮らしの狭いキッチンで本格的な自炊をする負担がなく、購入したその場ですぐに味わえる手軽さも、疲れた日の精神的なリフレッシュとして高く評価されています。日常の中に小さな **「選ぶ楽しさ」** と **「比較するワクワク感」** を組み込むことで、短時間でも濃密な幸福感を得られる点が、多忙な現代人のライフスタイルに強く合致しています。

### トピック2：コンビニ各社が仕掛ける秋の「品種・産地推し」スイーツ最前線
セブン-イレブン、ファミリーマート、ローソンなどの主要コンビニチェーンでは、毎年秋になると「さつまいも」「栗」「かぼちゃ」をテーマにしたフェアが目玉企画として定着しています。近年のトレンドとして顕著なのが、単なる「芋味」「栗味」といった大まかなフレーバー展開にとどまらず、 **「千葉県産紅はるか」** や **「熊本県産和栗」** のように具体的な品種や産地を前面に押し出した商品開発です。例えばセブン-イレブンでは濃厚な蜜芋テリーヌや産地指定プリン、ファミリーマートでは秋恒例のお芋スイーツ企画を展開するなど、各社が独自性を競い合っています。これらの商品は1点あたり **200円〜400円前後** という手頃な価格帯で購入できるため、働く単身層が仕事終わりに複数種類を買いそろえて自宅で食べ比べるハードルが非常に低くなっています。専門店レベルのクオリティを誇るスイーツを手軽にハシゴして比較できる環境は、まさに日常のすきま時間で楽しむ「食の推し活」の格好のフィールドとなっています。

### トピック3：焼き芋＆冷やし芋で味わう「さつまいも品種」の食感・甘み比較
コンビニのレジ横ホットスナックやチルドコーナーで定番化した焼き芋は、品種ごとの明確な個性や違いを実感できる最も身近な食べ比べアイテムです。現在の主流である **「紅はるか」** は、糖度が非常に高く加熱すると蜜があふれ出るようなねっとり濃厚な甘みが特徴で、強い甘さを求める層から絶大な支持を集めています。対して近年人気が急上昇している **「シルクスイート」** は、絹のように滑らかな舌触りと程よい水分量を持ち、上品ですっきりとした後味を楽しめるため、繊細な食感を好む人に選ばれています。さらに夏から秋にかけて冷蔵コーナーに並ぶ「冷やし焼き芋」も定着しており、冷やすことで甘みが凝縮されるとともに、食物繊維やレジスタントスターチを摂取できるヘルシーな間食としても評価されています。自分の好みが「ほくほく系」「ねっとり系」「しっとり系」のどれに当てはまるのかを突き詰めるプロセスそのものが、愛着の湧く **「推し芋探し」** の豊かな体験となります。

### トピック4：和栗 vs 洋栗！奥深いモンブランの世界とペアリングの楽しみ
秋スイーツの主役であるモンブランは、使用される栗のルーツによって風味や楽しみ方が全く異なる奥深いジャンルです。茨城県産や熊本県産などの **「和栗」** を使ったモンブランは、栗本来の繊細で素朴な風味やほくほくとした香りを生かすため、甘さを控えめにして緑茶やほうじ茶と合わせるスタイルが推奨されます。一方でフランス産やイタリア産などの **「洋栗」** を使ったモンブランは、ラム酒やバニラの香りを効かせた濃厚でクリーミーな味わいが特徴で、深煎りコーヒーや紅茶との相性が抜群です。コンビニやデパ地下の店頭では和栗と洋栗のモンブランが同時に並ぶことも多く、同じ栗スイーツでありながら対極にある味わいを手軽に比較検証できます。自宅でドリンクとのペアリングまでこだわり、香りの余韻や口どけの違いをじっくり吟味する時間は、忙しい平日の夜を上質なリラックスタイムへと変えてくれます。

### トピック5：タイパ良く楽しむ！単身者のための「手軽な食べ比べ＆記録ルーティン」
忙しい単身者が無理なく「食の推し活」を趣味として継続するためには、準備や後片付けの負担を最小限に抑える工夫が欠かせません。例えばスーパーやコンビニのカットフルーツ売り場を活用すれば、皮むきの手間がなく生ゴミも出ない **「シャインマスカット」** や **「クイーンルージュ」** などの高級ぶどうを手軽に少量ずつ食べ比べられます。食べ比べた感想は、スマートフォンのメモアプリや **SNS** を活用し、甘み・酸味・食感・リピート度を星の数でシンプルに記録するのがおすすめです。短文や写真1枚でサッと記録を残すルーティンを作ることで、情報が蓄積されて自分だけの **「秋の味覚データベース」** が完成していく達成感が得られます。大がかりな準備をせずとも、帰宅後のわずか **15分** で季節の移ろいと自分好みの味覚を深く味わう習慣は、働く世代の日常に心地よい彩りと充足感をもたらします。

**[writer_agent]**

仕事帰りにふらっと立ち寄る身近なコンビニで、自分だけの「推しの味」を探してみませんか？
今年の秋は、旬の味覚やスイーツの個性をとことん深掘りする「食の推し活」を新しい趣味にしてみるのがおすすめです。

## すきま時間で気軽に楽しむ「食の推し活」とは？

忙しい日々の中で大がかりなレジャーに出かけるのは少しハードルが高いものですが、「食の推し活」なら日常のすきま時間で手軽に満喫できます。

単に秋の味覚を口にするだけでなく、品種や産地の違いに目を向けて「自分の舌に一番しっくりくる味」を探求するのがこの趣味の面白さです。自炊の手間をかけずに購入してすぐ味わえるため、タイパやコスパを重視したい働く単身世代の気軽なご褒美として注目を集めています。日常の中にちょっとした「選ぶ楽しさ」を取り入れるだけで、いつもの夜がぐっと豊かな時間に変わります。

## ここまで進化した！秋の「品種・産地推し」最前線

最近のコンビニ各社では、単なる芋や栗といった大まかな味付けにとどまらず、素材のルーツにこだわった商品展開が大きなトレンドになっています。

- 千葉県産紅はるかを使った濃厚な蜜芋テリーヌや、熊本県産和栗を使用したプリンなど、専門店レベルのこだわりスイーツが200円〜400円前後で手に入ります。
- 定番の焼き芋コーナーでも、蜜があふれるねっとり濃厚な「紅はるか」と、絹のように滑らかで上品な「シルクスイート」が並び、冷やし焼き芋など季節に合わせた楽しみ方も定着しています。
- スイーツの王道であるモンブランも、素朴な香りを緑茶やほうじ茶と楽しむ「和栗」と、ラム酒が香るリッチな風味を深煎りコーヒーと合わせる「洋栗」の食べ比べが身近な店頭で手軽に試せます。

各チェーンをハシゴして気になる商品をいくつか買いそろえるだけでも、自宅ですぐに贅沢な比較検証が楽しめます。

## 帰宅後15分！スマホで残す「推し味メモ」のすすめ

食の推し活を心地よく続けるために、帰宅後のわずか15分でできる「スマホでの星付け記録」を試してみてはいかがでしょうか。

例えば、生ゴミや皮むきの手間がないカットフルーツ売り場のぶどうや、レジ横の焼き芋を2種類選んで買ってみます。食べ比べたら、スマートフォンのメモアプリやSNSを開き、「甘み」「食感」「リピート度」などを星の数でシンプルにメモしていくだけでOKです。

短い感想と写真を1枚残すルーティンを重ねていくだけで、自分だけの「秋の味覚データベース」が仕上がっていきます。面倒な準備や後片付けを気にせず、自分の好みが可視化されていく達成感をぜひ味わってみてください。

## 毎日の帰り道をちょっと特別な探求の時間に

「食の推し活」は、特別な準備やまとまったお金をかけなくても、日々の帰り道をワクワクする探究の時間へと変えてくれる新しい趣味です。

身近なお店で手に入る秋の恵みの中から、あなたの毎日に小さな幸せを運んでくれる「推しの逸品」をぜひ見つけてみてくださいね。

**[reviewer_agent]**

原稿の執筆、お疲れ様でした。
ご提示いただいた原稿について、レビュー結果と改善に向けたコメントをお伝えします。

---

### 記事の良い点

- **斬新で魅力的なテーマ設定：**
  流行りの「推し活」を遠征やグッズ収集ではなく「秋の身近な食（品種・産地）」にフォーカスした切り口が非常に新しく、読者の興味を惹きつけます。
- **ターゲット層への高い解像度：**
  20代〜30代の多忙な単身世代に向けて、タイパ・コスパや後片付けの負担軽減（コンビニ活用、カットフルーツ、スマホ記録など）に配慮された現実的な提案ができています。
- **味わいの対比が具体的でイメージしやすい：**
  「紅はるか（ねっとり濃厚） vs シルクスイート（滑らか上品）」「和栗（ほうじ茶・素朴） vs 洋栗（深煎りコーヒー・リッチ）」など、味の特徴やペアリングまで具体的に描写されており、読者が実際に試してみたくなる魅力があります。

---

### 修正ポイント

規定の条件を満たし、より読者に刺さる記事に仕上げるために、以下の点について修正・追記をお願いします。

- **1. 記事タイトルの新設（文字数とマークダウン記法）：**
  - 現在、記事の冒頭にタイトルが設定されていません。
  - マークダウンヘッダ **`#`** を使用し、 **40文字程度** でタイトルをつけてください。
  - 「秋の味覚が本格化する今、今日からすぐに試してみたい」と読者が感じるような、即効性と生活への役立ち感を押し出したフレーズを含めましょう。
  - *（タイトル例案：`# 今夜の帰り道からすぐ試せる！秋のコンビニで始める「食の推し活」入門ガイド` など）*

- **2. 導入部への「なぜいま取り上げるのか」の動機づけの強化：**
  - タイトル直後の導入部で、「なぜ **“いま（この秋）”** このテーマを始めるべきなのか」の理由を明確に記述してください。
  - 「各コンビニが秋限定の品種フェアに本気を出している今の時期だからこそ、専門店に行かなくても最高の食べ比べができる」といった時節の urgency（緊急性・今読むべき理由）を補強すると、読者のモチベーションがグッと高まります。

- **3. 各セクションのサブタイトルに絵文字を追加：**
  - 条件にある通り、各見出し（`##`）に内容に合わせた絵文字を配置して、親しみやすく視覚的にも楽しい雰囲気を演出してください。
  - *（例：`## 🍠 すきま時間で気軽に楽しむ「食の推し活」とは？` / `## 🌰 ここまで進化した！秋の「品種・産地推し」最前線` など）*

- **4. 「今日から実践できる具体例」を3つ以上、明確に提示：**
  - 原稿内にも素晴らしいアイデア（焼き芋比較、モンブラン×ドリンクのペアリング、スマホ記録など）が含まれていますが、現状は解説文の中に溶け込んでいます。
  - 「今日からできる！おすすめの実践アクション3選」のように枠組みを整理し、読者が仕事帰りにそのまま真似できる具体的なアクションとして3点以上、分かりやすく提示してください。
  - *（アクションの例：①「レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる」、②「モンブランとお茶・コーヒーのペアリングを試す」、③「スマホのメモ帳に星3つで味覚レビューをつける」など）*

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？

In [17]:
query = '''
書き直してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事の書き直しを依頼します。

**[writer_agent]**

# 今夜の帰り道からすぐ試せる！秋のコンビニで始める「食の推し活」入門ガイド

仕事帰りにふらっと立ち寄る身近なコンビニで、自分だけの「推しの味」を探してみませんか？
主要チェーンが秋限定の品種フェアに最も熱を入れている今の時期だからこそ、遠出や専門店巡りをしなくても、身近な場所で最高峰の食べ比べを満喫できる絶好のチャンスです。

## 🍠 すきま時間で気軽に楽しむ「食の推し活」とは？

忙しい日々の中で大がかりなレジャーに出かけるのは少しハードルが高いものですが、「食の推し活」なら日常のすきま時間で手軽に満喫できます。

単に秋の味覚を口にするだけでなく、品種や産地の違いに目を向けて「自分の舌に一番しっくりくる味」を探求するのがこの趣味の醍醐味です。自炊の負担がなく、1点あたり数百円程度で手に入るため、タイパやコスパを重視したい働く単身世代のご褒美として注目を集めています。日常の中に小さな「選ぶ楽しさ」と「比較するワクワク感」を取り入れるだけで、いつもの夜が濃密なリフレッシュタイムに変わります。

## 🌰 ここまで進化した！秋の「品種・産地推し」最前線

最近のコンビニ各社では、単なる芋や栗といったフレーバー展開にとどまらず、素材のルーツを前面に押し出した本格スイーツが棚を賑わせています。

セブン-イレブンやファミリーマートをはじめ、千葉県産紅はるかを使った蜜芋テリーヌや熊本県産和栗のスイーツなど、専門店レベルのこだわり商品が200円〜400円前後という手頃な価格帯で登場しています。

さらに、チルドコーナーやレジ横の定番となった焼き芋やモンブランも、素材や製法による違いがより明確になりました。蜜たっぷりで濃厚な甘みの品種と上品で滑らかな舌触りの品種、あるいは繊細な風味の和栗と洋酒が効いた洋栗など、対極にある味わいを手軽にハシゴして比較できる環境が整っています。

## 📝 今夜から真似できる！「食の推し活」実践アクション3選

忙しい平日の夜でも、準備や後片付けの負担をかけずにすぐ試せるおすすめのアクションを3つご紹介します。

- アクション1：レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる
蜜があふれる濃厚な甘さの「紅はるか」と、絹のように滑らかな舌触りですっきりとした後味の「シルクスイート」を買いそろえてみましょう。温かいまま食べるのはもちろん、冷蔵コーナーの「冷やし焼き芋」を選べば、甘みが凝縮されたヘルシーなスイーツとしても楽しめます。

- アクション2：モンブラン×ドリンクのペアリングを試す
素朴で繊細な香りが魅力の「和栗モンブラン」にはすっきりとした緑茶やほうじ茶を、バニラやラム酒が香るリッチな「洋栗モンブラン」には深煎りコーヒーや紅茶を合わせてみます。飲み物との組み合わせによって引き出される風味の違いをじっくり吟味するだけで、贅沢なカフェ気分が味わえます。

- アクション3：スマホのメモ帳やSNSに星3つで味覚レビューをつける
食べ比べた感想は、スマホのメモアプリなどに「甘み」「食感」「リピート度」などを星の数でサッと記録しておきます。皮むき不要で生ゴミも出ないカットフルーツの高級ぶどう（シャインマスカットやクイーンルージュなど）から始めるのも手軽です。写真1枚と短文の記録が蓄積されることで、自分だけの「秋の味覚データベース」を作る達成感が得られます。

## 🍁 毎日の帰り道をちょっと特別な探求の時間に

「食の推し活」は、特別な準備やまとまった時間をかけなくても、いつもの帰り道をワクワクする探究の時間へと変えてくれる新しい趣味です。

今まさに旬を迎えているコンビニの棚から、あなたの毎日に心地よい彩りを添えてくれる「推しの逸品」をぜひ見つけ出してみてくださいね。

**[reviewer_agent]**

修正稿の作成、お疲れ様でした。
前回のレビューで指摘した4つの条件がすべて的確に反映され、記事としての完成度と説得力が大幅に向上しています。

直近の原稿内容を踏まえ、レビュー結果をお伝えします。

---

### 記事の良い点

- **条件に合致した魅力的なタイトル：**
  文字数（37文字）やマークダウン形式（ `#` ）の指定を満たしつつ、「今夜の帰り道からすぐ試せる！」という即効性と生活への役立ち感がしっかり伝わるタイトルになっています。
- **「いま読むべき理由」が明確になった導入部：**
  主要コンビニ各社が秋の品種フェアに注力している時節性と結びつけたことで、読者が「今すぐ試したい」と感じる強い動機づけができています。
- **絵文字による親しみやすさの演出：**
  各セクションの見出し（ `##` ）にテーマと連動した絵文字（🍠・🌰・📝・🍁）が配置され、視覚的な楽しさと読みやすさが格段にアップしました。
- **今日から真似できるアクション3選の明確化：**
  「焼き芋食べ比べ」「モンブランのペアリング」「星付け記録」と、読者が仕事帰りにそのまま再現できる実践例が分かりやすく整理されており、実用的な生活情報記事として非常に質の高い構成になりました。

---

### 修正・ブラッシュアップのポイント

指定された必須条件はすべてクリアできています。
さらに記事を読みやすくし、読者の行動を促すための **「プラスアルファの改善提案」** として、以下の点をご検討ください。

- **アクションリストの視覚的な強調（可読性の向上）：**
  - 「実践アクション3選」の箇条書き部分で、アクションの見出し（例： **アクション1：レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる** ）を太字にしたり、説明文との間に改行を入れたりすると、スマートフォン等の画面でもさらにメリハリがついて読みやすくなります。

- **実践ハードルを下げる「目安（時間・予算）」の添え字：**
  - 各アクションの冒頭に、例えば **「（所要時間：10分 / 予算：約400円）」** のような目安を軽く添えると、忙しい単身読者が「これなら自分にもすぐできる」と一層ハードルを低く感じられます。

- **文末表現の微調整：**
  - 全体を通して非常に心地よいトーン＆マナーですが、最後のまとめの段落にもう一言「今夜、最寄りのコンビニに寄り道してみませんか？」といった、今夜の行動を直接後押しする呼びかけを添えると、記事を読んだ後の読者のアクション率がさらに高まります。

---

全体として、読者のライフスタイルに寄り添った非常に魅力的な記事に仕上がっています！

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？「はい」か「いいえ」でお答えください。

In [18]:
query = '''
はい。もう一度、書き直してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事の書き直しを依頼します。

**[writer_agent]**

# 今夜の帰り道からすぐ試せる！秋のコンビニで始める「食の推し活」入門ガイド

仕事帰りにふらっと立ち寄る身近なコンビニで、自分だけの「推しの味」を探してみませんか？
主要チェーンが秋限定の品種フェアに最も熱を入れている今の時期だからこそ、遠出や専門店巡りをしなくても、身近な場所で最高峰の食べ比べを満喫できる絶好のチャンスです。

## 🍠 すきま時間で気軽に楽しむ「食の推し活」とは？

忙しい日々の中で大がかりなレジャーに出かけるのは少しハードルが高いものですが、「食の推し活」なら日常のすきま時間で手軽に満喫できます。

単に秋の味覚を口にするだけでなく、品種や産地の違いに目を向けて「自分の舌に一番しっくりくる味」を探求するのがこの趣味の醍醐味です。自炊の負担がなく、1点あたり数百円程度で手に入るため、タイパやコスパを重視したい働く単身世代のご褒美として注目を集めています。日常の中に小さな「選ぶ楽しさ」と「比較するワクワク感」を取り入れるだけで、いつもの夜が濃密なリフレッシュタイムに変わります。

## 🌰 ここまで進化した！秋の「品種・産地推し」最前線

最近のコンビニ各社では、単なる芋や栗といったフレーバー展開にとどまらず、素材のルーツを前面に押し出した本格スイーツが棚を賑わせています。

セブン-イレブンやファミリーマートをはじめ、千葉県産紅はるかを使った蜜芋テリーヌや熊本県産和栗のスイーツなど、専門店レベルのこだわり商品が200円〜400円前後という手頃な価格帯で登場しています。

さらに、チルドコーナーやレジ横の定番となった焼き芋やモンブランも、素材や製法による違いがより明確になりました。蜜たっぷりで濃厚な甘みの品種と上品で滑らかな舌触りの品種、あるいは繊細な風味の和栗と洋酒が効いた洋栗など、対極にある味わいを手軽にハシゴして比較できる環境が整っています。

## 📝 今夜から真似できる！「食の推し活」実践アクション3選

忙しい平日の夜でも、準備や後片付けの負担をかけずにすぐ試せるおすすめのアクションを3つご紹介します。

- **アクション1：レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる** （所要時間：約10分 / 予算：約400円〜）

  蜜があふれる濃厚な甘さの「紅はるか」と、絹のように滑らかな舌触りですっきりとした後味の「シルクスイート」を買いそろえてみましょう。温かいまま食べるのはもちろん、冷蔵コーナーの「冷やし焼き芋」を選べば、甘みが凝縮されたヘルシーなスイーツとしても楽しめます。

- **アクション2：モンブラン×ドリンクのペアリングを試す** （所要時間：約15分 / 予算：約400円〜）

  素朴で繊細な香りが魅力の「和栗モンブラン」にはすっきりとした緑茶やほうじ茶を、バニラやラム酒が香るリッチな「洋栗モンブラン」には深煎りコーヒーや紅茶を合わせてみます。飲み物との組み合わせによって引き出される風味の違いをじっくり吟味するだけで、贅沢なカフェ気分が味わえます。

- **アクション3：スマホのメモ帳やSNSに星3つで味覚レビューをつける** （所要時間：約5分 / 予算：0円〜）

  食べ比べた感想は、スマホのメモアプリなどに「甘み」「食感」「リピート度」などを星の数でサッと記録しておきます。皮むき不要で生ゴミも出ないカットフルーツの高級ぶどう（シャインマスカットやクイーンルージュなど）から始めるのも手軽です。写真1枚と短文の記録が蓄積されることで、自分だけの「秋の味覚データベース」を作る達成感が得られます。

## 🍁 毎日の帰り道をちょっと特別な探求の時間に

「食の推し活」は、特別な準備やまとまった時間をかけなくても、いつもの帰り道をワクワクする探究の時間へと変えてくれる新しい趣味です。

今まさに旬を迎えているコンビニの棚から、あなたの毎日に心地よい彩りを添えてくれる「推しの逸品」をぜひ見つけ出してみてください。今夜、仕事帰りに最寄りのコンビニへ少し寄り道してみませんか？

**[reviewer_agent]**

原稿の再推敲、お疲れ様でした。
前回のレビューで提案した「視覚的なメリハリ」「目安（所要時間・予算）の追記」「最後のひと押しとなる呼びかけ」がすべて完璧に反映され、完成度が極めて高い原稿に仕上がっています。

これまでの履歴も踏まえ、直近の原稿に対する最終レビューをお伝えします。

---

### 記事の良い点

- **条件の完全な網羅と高い完成度：**
  文字数（37文字）やマークダウンヘッダ（ `#` ）の指定、導入部における時節性の動機づけ、各見出しの絵文字、3つの具体的アクション提示など、すべての指定条件が隙なくクリアされています。
- **抜群の可読性とレイアウト配慮：**
  実践アクションの見出しを太字にし、説明文との間に適切な改行を入れたことで、スマートフォン等の縦スクロール画面でもストレスなく要点が飛び込んでくる素晴らしいレイアウトになりました。
- **読者の実行ハードルを下げる工夫：**
  各アクションに「所要時間」と「予算目安」が添えられたことで、多忙な働く世代が「これなら今夜すぐできる」と具体的にイメージできるようになりました。
- **行動を後押しする自然なクロージング：**
  結びの一文（「今夜、仕事帰りに最寄りのコンビニへ少し寄り道してみませんか？」）によって、読後すぐにコンビニへ立ち寄りたくなる心理的誘導が非常にスマートに決まっています。

---

### 修正・公開に向けたチェックポイント

記事本文としては**修正すべき必須項目はなく、このまま公開・納品可能な完成度**に達しています。
メディア掲載やSNS展開にあたって、さらに読者を惹きつけるための **「公開時の最終チェックポイント」** を箇条書きで記載します。

- **アイキャッチ画像・サムネイルの選定：**
  - 記事冒頭やSNSシェア時の画像には、文字情報（「今夜試せる」「秋のコンビニ推し活」など）を乗せた焼き芋やモンブランの高解像度写真を選ぶと、タイトルの引きの強さと相乗効果を生み出せます。

- **メタディスクリプション（抜粋文）の作成：**
  - 検索エンジンやSNS表示用の抜粋文を用意する場合は、導入部の2文（「仕事帰りにふらっと立ち寄る身近なコンビニで〜絶好のチャンスです。」）をベースに設定すると、要点が簡潔に伝わりクリック率を高められます。

- **リンク設定（記事展開時のアイデア）：**
  - 掲載プラットフォームで許容されている場合、各アクション内で言及している「紅はるか」や「シルクスイート」「和栗・洋栗」に関連する過去記事や特集ページへの内部リンクを設けると、サイト内回遊率の向上にもつながります。

---

非常に魅力的で読者目線に立った素晴らしい読み物記事になりました。執筆対応ありがとうございました！

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？

In [19]:
query = '''
これで完成です。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事が完成しました。

**[article_writing_workflow]**


### 完成した記事を表示します。

---

# 今夜の帰り道からすぐ試せる！秋のコンビニで始める「食の推し活」入門ガイド

仕事帰りにふらっと立ち寄る身近なコンビニで、自分だけの「推しの味」を探してみませんか？
主要チェーンが秋限定の品種フェアに最も熱を入れている今の時期だからこそ、遠出や専門店巡りをしなくても、身近な場所で最高峰の食べ比べを満喫できる絶好のチャンスです。

## 🍠 すきま時間で気軽に楽しむ「食の推し活」とは？

忙しい日々の中で大がかりなレジャーに出かけるのは少しハードルが高いものですが、「食の推し活」なら日常のすきま時間で手軽に満喫できます。

単に秋の味覚を口にするだけでなく、品種や産地の違いに目を向けて「自分の舌に一番しっくりくる味」を探求するのがこの趣味の醍醐味です。自炊の負担がなく、1点あたり数百円程度で手に入るため、タイパやコスパを重視したい働く単身世代のご褒美として注目を集めています。日常の中に小さな「選ぶ楽しさ」と「比較するワクワク感」を取り入れるだけで、いつもの夜が濃密なリフレッシュタイムに変わります。

## 🌰 ここまで進化した！秋の「品種・産地推し」最前線

最近のコンビニ各社では、単なる芋や栗といったフレーバー展開にとどまらず、素材のルーツを前面に押し出した本格スイーツが棚を賑わせています。

セブン-イレブンやファミリーマートをはじめ、千葉県産紅はるかを使った蜜芋テリーヌや熊本県産和栗のスイーツなど、専門店レベルのこだわり商品が200円〜400円前後という手頃な価格帯で登場しています。

さらに、チルドコーナーやレジ横の定番となった焼き芋やモンブランも、素材や製法による違いがより明確になりました。蜜たっぷりで濃厚な甘みの品種と上品で滑らかな舌触りの品種、あるいは繊細な風味の和栗と洋酒が効いた洋栗など、対極にある味わいを手軽にハシゴして比較できる環境が整っています。

## 📝 今夜から真似できる！「食の推し活」実践アクション3選

忙しい平日の夜でも、準備や後片付けの負担をかけずにすぐ試せるおすすめのアクションを3つご紹介します。

- **アクション1：レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる** （所要時間：約10分 / 予算：約400円〜）

  蜜があふれる濃厚な甘さの「紅はるか」と、絹のように滑らかな舌触りですっきりとした後味の「シルクスイート」を買いそろえてみましょう。温かいまま食べるのはもちろん、冷蔵コーナーの「冷やし焼き芋」を選べば、甘みが凝縮されたヘルシーなスイーツとしても楽しめます。

- **アクション2：モンブラン×ドリンクのペアリングを試す** （所要時間：約15分 / 予算：約400円〜）

  素朴で繊細な香りが魅力の「和栗モンブラン」にはすっきりとした緑茶やほうじ茶を、バニラやラム酒が香るリッチな「洋栗モンブラン」には深煎りコーヒーや紅茶を合わせてみます。飲み物との組み合わせによって引き出される風味の違いをじっくり吟味するだけで、贅沢なカフェ気分が味わえます。

- **アクション3：スマホのメモ帳やSNSに星3つで味覚レビューをつける** （所要時間：約5分 / 予算：0円〜）

  食べ比べた感想は、スマホのメモアプリなどに「甘み」「食感」「リピート度」などを星の数でサッと記録しておきます。皮むき不要で生ゴミも出ないカットフルーツの高級ぶどう（シャインマスカットやクイーンルージュなど）から始めるのも手軽です。写真1枚と短文の記録が蓄積されることで、自分だけの「秋の味覚データベース」を作る達成感が得られます。

## 🍁 毎日の帰り道をちょっと特別な探求の時間に

「食の推し活」は、特別な準備やまとまった時間をかけなくても、いつもの帰り道をワクワクする探究の時間へと変えてくれる新しい趣味です。

今まさに旬を迎えているコンビニの棚から、あなたの毎日に心地よい彩りを添えてくれる「推しの逸品」をぜひ見つけ出してみてください。今夜、仕事帰りに最寄りのコンビニへ少し寄り道してみませんか？
    

**[AWW-15]**

セッションのステートに格納された完成原稿のマークダウンテキストを取得します。

In [20]:
session = await chat_client.adk_app.async_get_session(
    user_id = chat_client.user_id,
    session_id = chat_client.session_id,
)

print(session.state['article_content'])

# 今夜の帰り道からすぐ試せる！秋のコンビニで始める「食の推し活」入門ガイド

仕事帰りにふらっと立ち寄る身近なコンビニで、自分だけの「推しの味」を探してみませんか？
主要チェーンが秋限定の品種フェアに最も熱を入れている今の時期だからこそ、遠出や専門店巡りをしなくても、身近な場所で最高峰の食べ比べを満喫できる絶好のチャンスです。

## 🍠 すきま時間で気軽に楽しむ「食の推し活」とは？

忙しい日々の中で大がかりなレジャーに出かけるのは少しハードルが高いものですが、「食の推し活」なら日常のすきま時間で手軽に満喫できます。

単に秋の味覚を口にするだけでなく、品種や産地の違いに目を向けて「自分の舌に一番しっくりくる味」を探求するのがこの趣味の醍醐味です。自炊の負担がなく、1点あたり数百円程度で手に入るため、タイパやコスパを重視したい働く単身世代のご褒美として注目を集めています。日常の中に小さな「選ぶ楽しさ」と「比較するワクワク感」を取り入れるだけで、いつもの夜が濃密なリフレッシュタイムに変わります。

## 🌰 ここまで進化した！秋の「品種・産地推し」最前線

最近のコンビニ各社では、単なる芋や栗といったフレーバー展開にとどまらず、素材のルーツを前面に押し出した本格スイーツが棚を賑わせています。

セブン-イレブンやファミリーマートをはじめ、千葉県産紅はるかを使った蜜芋テリーヌや熊本県産和栗のスイーツなど、専門店レベルのこだわり商品が200円〜400円前後という手頃な価格帯で登場しています。

さらに、チルドコーナーやレジ横の定番となった焼き芋やモンブランも、素材や製法による違いがより明確になりました。蜜たっぷりで濃厚な甘みの品種と上品で滑らかな舌触りの品種、あるいは繊細な風味の和栗と洋酒が効いた洋栗など、対極にある味わいを手軽にハシゴして比較できる環境が整っています。

## 📝 今夜から真似できる！「食の推し活」実践アクション3選

忙しい平日の夜でも、準備や後片付けの負担をかけずにすぐ試せるおすすめのアクションを3つご紹介します。

- **アクション1：レジ横の焼き芋2種（ねっとり系・しっとり系）を食べ比べる** （所要時間：約10分 / 予算：約400円〜）

  蜜があふれる濃厚な甘さの「紅はるか」と、絹のように滑らかな舌触りですっきりとした後味の「シルク